# Taller: Agentes al rescate

Reproduce las decisiones guardadas del agente, modifica una regla de priorización y revisa casos. Esta versión no llama a una API de IA ni requiere credenciales. Puedes guardar una copia en Drive o descargar el cuaderno desde el menú Archivo. Uso educativo: ninguna salida sustituye la revisión clínica.

## Experimento original

Modelo: `gpt-6-sol`; prompt: `agent_v2`. Entrada: frontal, edad aproximada, sexo y hora asignada. Salida: prioridad, puntaje, hallazgos y motivo. El orden se calcula en código. El prompt original decía 17:45; el relato del taller se corrigió a 17:50 sin repetir la inferencia.

In [ ]:
import json, urllib.request
from collections import Counter
URL = 'https://agentes.hshenriquez.workers.dev/downloads/datos_taller.json'
with urllib.request.urlopen(URL, timeout=30) as r: datos = json.load(r)
casos = datos['cases']
resultados = {r['id']: r for r in datos['resultados']}
orden_llegada = [c['id'] for c in casos]
print(len(casos), 'casos', dict(Counter(r['prioridad'] for r in resultados.values())))

## 1. Reproduce el orden y lee el prompt

La categoría manda; luego el puntaje; los empates conservan el orden de llegada.

In [ ]:
print(datos['prompt_original'])
rango = {'critico': 0, 'urgente': 1, 'rutina': 2}
orden = sorted(orden_llegada, key=lambda cid: (rango[resultados[cid]['prioridad']], -resultados[cid]['puntaje'], orden_llegada.index(cid)))
assert orden == datos['orden_propuesto']
for n,cid in enumerate(orden,1):
 r=resultados[cid]
 print(f"{n:2d} {cid} {r['prioridad']:8s} {r['puntaje']:5.1f} {r['mensaje']}")

## 2. Cambia una regla local

Prueba incluir `urgente` en `prioridades_hoy` o modifica los puntajes mínimos. Esto reinterpreta resultados guardados; no cambia el prompt ni genera una nueva inferencia.

In [ ]:
prioridades_hoy = {'critico'}  # prueba {'critico','urgente'}
minimo = {'critico':0, 'urgente':50, 'rutina':101}
seleccion = [cid for cid in orden if resultados[cid]['prioridad'] in prioridades_hoy and resultados[cid]['puntaje'] >= minimo[resultados[cid]['prioridad']]]
base = {cid for cid,r in resultados.items() if r['prioridad']=='critico'}
print('Para informar hoy:', seleccion)
print('Añadidos:', sorted(set(seleccion)-base), 'Retirados:', sorted(base-set(seleccion)))

## 3. Revisa un caso y sus imágenes

Cambia `caso_id` y contrasta la propuesta con tu decisión del taller. El ZIP contiene las 30 frontales utilizadas y un manifiesto SHA-256; se descarga directamente desde Cloudflare. Esta sección tampoco llama a una API de IA.

In [ ]:
caso_id = 'c16'
print(json.dumps(resultados[caso_id], ensure_ascii=False, indent=2))
import io, hashlib, zipfile
from PIL import Image
from IPython.display import display
ZIP_URL = 'https://agentes.hshenriquez.workers.dev/downloads/rx_frontales_30.zip'
with urllib.request.urlopen(ZIP_URL, timeout=120) as r: paquete = zipfile.ZipFile(io.BytesIO(r.read()))
assert len([n for n in paquete.namelist() if n.startswith('imagenes/') and n.endswith('.png')]) == 30
for item in datos['imagenes']:
 assert hashlib.sha256(paquete.read(item['archivo'])).hexdigest() == item['sha256']
print('30 frontales verificadas')
display(Image.open(io.BytesIO(paquete.read(f'imagenes/{caso_id}_frontal.png'))))

## Para continuar

El siguiente paso será permitir que cada asistente ejecute un modelo con su propia API key y compare esa nueva corrida con la guardada. La clave deberá introducirse en su sesión privada, nunca en el cuaderno compartido. Para un flujo real, define el protocolo local, la acción propuesta en el visualizador y la revisión por un radiólogo.